# individual_ne_regressions.ipynb\n\n**Purpose:** Estimate individual-level slopes of Neural Efficiency over sessions\nand compare the distribution of slopes between groups using Fisher exact test\nand independent-samples t-test.\n\n**Inputs:**\n- nirs_neural_efficiency.xlsx — NE index per subject × session × ROI\n\n**Outputs:**\n- individual_ne_slopes.xlsx — slope, p-value, R² per subject × ROI\n- Scatter/regression plots (PNG)\n\n**Method:**\n- OLS: NE ~ sessao_num per subject × ROI (sessions 1-9)\n- Fisher exact: proportion of subjects with positive slope per group\n- Two-sample t-test: mean slope accelerated vs non-accelerated\n\n**Library:** scipy, statsmodels (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Individual Neural Efficiency Regressions
**Project SESI | Input: fnirs_neural_efficiency.xlsx**

- Individual OLS slope per subject × ROI
- FacetGrid plot: individual regression lines by group and ROI
- Group comparison: Fisher exact + t-test on slopes

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact, ttest_ind
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration and load

In [ ]:
# Update this path to match your local data directory
PATH_IN        = r'../data/fnirs_neural_efficiency.xlsx'
# Update this path to match your local data directory
PATH_OUT_PLOT  = r'../results/individual_ne_regressions.png'
# Update this path to match your local data directory
PATH_OUT_EXCEL = r'../results/individual_ne_slopes.xlsx'

GROUP_LABELS = {
    'acelerado'    : 'Accelerated',
    'nao_acelerado': 'Non-Accelerated'
}

df = pd.read_excel(PATH_IN)

# Map group labels for plot
df['group_label'] = df['group'].map(GROUP_LABELS)

print(f'Shape          : {df.shape}')
print(f'Subjects       : {df["subject"].nunique()}')
print(f'Sessions       : {sorted(df["sessao_num"].unique())}')
print(f'ROIs           : {df["roi"].unique().tolist()}')
print(f'Groups         : {df["group"].unique().tolist()}')

## 2 — Individual OLS regressions per subject × ROI

In [ ]:
def run_individual_regressions(dataframe):
    """OLS slope of neural_efficiency ~ sessao_num for each subject × ROI."""
    results = []
    for subject in dataframe['subject'].unique():
        for roi in dataframe['roi'].unique():
            subset = dataframe[
                (dataframe['subject'] == subject) &
                (dataframe['roi'] == roi)
            ]
            if len(subset) < 3:
                continue
            model = smf.ols('neural_efficiency ~ sessao_num', data=subset).fit()
            results.append({
                'subject'  : subject,
                'group'    : subset['group'].iloc[0],
                'roi'      : roi,
                'slope'    : model.params.get('sessao_num', np.nan),
                'pvalue'   : model.pvalues.get('sessao_num', np.nan),
                'r_squared': model.rsquared
            })

    df_res = pd.DataFrame(results)

    def sig_label(p):
        if p < 0.05:  return 'p < 0.05'
        if p < 0.10:  return 'p < 0.10'
        return 'n.s.'

    df_res['significance'] = df_res['pvalue'].apply(sig_label)
    return df_res.sort_values(['group','subject','roi']).reset_index(drop=True)


df_slopes = run_individual_regressions(df)

print('Individual regression results:')
print(df_slopes.groupby(['group','roi'])['slope'].describe().round(3))
print('\nSignificant slopes (p < 0.05) by group and ROI:')
print(df_slopes.groupby(['group','roi'])['significance']
      .apply(lambda x: (x == 'p < 0.05').sum())
      .rename('n_significant'))

## 3 — FacetGrid: individual regression lines

In [ ]:
sns.set_theme(style='whitegrid')

g = sns.lmplot(
    data=df,
    x='sessao_num',
    y='neural_efficiency',
    hue='subject',
    col='roi',
    row='group_label',
    ci=None,
    palette='tab10',
    height=5,
    aspect=1.6,
    scatter_kws={'alpha': 0.5, 's': 35},
    line_kws={'alpha': 0.9, 'linewidth': 2.5}
)

g.fig.suptitle('Individual Regressions: Neural Efficiency Slopes',
               y=1.03, fontsize=15, fontweight='bold')
g.set_axis_labels('Session', 'Neural Efficiency', fontsize=12)
g.set_titles(col_template='{col_name}', row_template='{row_name}')

for ax in g.axes.flat:
    ax.set_xticks(sorted(df['sessao_num'].unique()))
    ax.axhline(0, linestyle='--', color='grey', linewidth=1, alpha=0.6)
    ax.grid(True, linestyle=':', alpha=0.7)

plt.savefig(PATH_OUT_PLOT, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_PLOT}')
plt.show()

## 4 — Group comparison: Fisher exact + t-test on slopes

In [ ]:
df_slopes['is_significant'] = (df_slopes['pvalue'] < 0.05).astype(int)

comparison_results = []

for roi in df_slopes['roi'].unique():
    subset = df_slopes[df_slopes['roi'] == roi]

    acc  = subset[subset['group'] == 'acelerado']
    ctrl = subset[subset['group'] == 'nao_acelerado']

    # Fisher exact — proportion of significant slopes
    tabela = pd.crosstab(subset['group'], subset['is_significant'])
    for col in [0, 1]:
        if col not in tabela.columns:
            tabela[col] = 0
    tabela = tabela[[0, 1]]
    _, p_fisher = fisher_exact(tabela)

    # t-test — slope magnitude
    t_stat, p_ttest = ttest_ind(
        acc['slope'], ctrl['slope'], nan_policy='omit'
    )

    # Proportion significant per group
    pct_acc  = acc['is_significant'].mean() * 100
    pct_ctrl = ctrl['is_significant'].mean() * 100

    comparison_results.append({
        'roi'                        : roi,
        'n_accelerated'              : len(acc),
        'slope_mean_accelerated'     : round(acc['slope'].mean(), 4),
        'sig_pct_accelerated'        : f'{pct_acc:.1f}%',
        'n_non_accelerated'          : len(ctrl),
        'slope_mean_non_accelerated' : round(ctrl['slope'].mean(), 4),
        'sig_pct_non_accelerated'    : f'{pct_ctrl:.1f}%',
        'p_fisher'                   : round(p_fisher, 4),
        'p_ttest'                    : round(p_ttest, 4),
        'interpretation'             : 'Group difference' if (p_ttest < 0.05 or p_fisher < 0.05) else 'No difference'
    })

df_comparison = pd.DataFrame(comparison_results)
print('Group Comparison — Slopes')
print('='*70)
print(df_comparison.to_string(index=False))

## 5 — Export

In [ ]:
with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_slopes[df_slopes['group'] == 'acelerado'].to_excel(
        writer, sheet_name='Accelerated', index=False)
    df_slopes[df_slopes['group'] == 'nao_acelerado'].to_excel(
        writer, sheet_name='Non_Accelerated', index=False)
    df_slopes.to_excel(
        writer, sheet_name='All_subjects', index=False)
    df_comparison.to_excel(
        writer, sheet_name='Group_comparison', index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print('   Sheets: Accelerated | Non_Accelerated | All_subjects | Group_comparison')